# 03 — ONNX Runtime Inference

## Objective
Run the exported BERT sequence classification model using ONNX Runtime.
Understand inference sessions, execution providers, input/output tensors and compare ONNX predictions against the original PyTorch model to verify numerical consistency.

### Load the ONNX model

In [ ]:
from pathlib import Path

import numpy as np
import onnxruntime as ort

MODEL_ID = "textattack/bert-base-uncased-SST-2"

MODEL_DIR = Path.cwd().parent / "models" / "bert-sst2-onnx"
ONNX_PATH = MODEL_DIR / "model.onnx"

print("Model path:", ONNX_PATH.resolve())
print("Model exists:", ONNX_PATH.exists())

In the `02_onnx_export.ipynb` notebook we used a `onnx` library to load and validate the model structure. In this scenario, we make some calculations using `onnxruntime` library.

### Run ONNX Inference

In [2]:
session = ort.InferenceSession(
    str(ONNX_PATH),
    providers=["CPUExecutionProvider"]
)

- Inference session is a object which represents a loaded ONNX model, ready to execute an inference process.
- Execution Provider describe which environment will execute the model process.

**For instance**:

`CPUExecutionProvider` = CPU processes

`CUDAExecutionProvider` = NVIDIA CUDA

`TensorrtExecutionProvider` = NVIDIA TensorRT

... executions.

First, we have to check the inference correctness.

In [ ]:
print(ort.get_all_providers())
print(session.get_providers())

### Check model interface

In the `02_onnx_export.ipynb` notebook we analyzed the saved graph in ONNX, now want to check the loaded session interface.

In [ ]:
for model_input in session.get_inputs():
    print(
        model_input.name,
        model_input.shape,
        model_input.type
    )

`get_inputs` returns a list of **NodeArg** objects, which describe a model inputs.

Each of them shares:
- name - name of input
- shape - expected shape
- type - data type

Likewise, we analyze the outputs.

In [ ]:
for model_output in session.get_outputs():
    print(
        model_output.name,
        model_output.shape,
        model_output.type
    )

### Prepare Sentences -> input data

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

sentences = [
    "I really enjoyed this movie.",
    "The movie was boring and disappointing",
    "The acting was excellent, but the story was weak."
]

encoded = tokenizer(
    sentences,
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="np"
)

for name, value in encoded.items():
    print(name, value.shape, value.dtype)

### Execute an inference process

In [ ]:
input_names = [item.name for item in session.get_inputs()]

ort_inputs = {
    name: encoded[name]
    for name in input_names
}

outputs = session.run(
    None,
    ort_inputs
)

logits = outputs[0]

print(logits)
print(logits.shape)

### From logits to prediction

In [ ]:
predictions = np.argmax(logits, axis=-1)

for sentence, prediction in zip(sentences, predictions):
    print(sentence)
    print("Prediction:", prediction)
    print()

### Comparison of PyTorch and ONNX Runtime

The main experiment in this notebook is to check whether the exported model kept its behavior.

In [ ]:
import torch

from transformers import AutoModelForSequenceClassification


pt_model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID)

pt_model.eval()

pt_inputs = tokenizer(
    sentences,
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

with torch.inference_mode():
    pt_outputs = pt_model(**pt_inputs)

pt_logits = pt_outputs.logits.numpy()

print("PyTorch logits:")
print(pt_logits)

print("\nONNX logits:")
print(logits)

difference = np.abs(pt_logits - logits)
print("Maximum absolute difference:", difference.max())
print("Mean absolute difference:", difference.mean())

is_close = np.allclose(
    pt_logits,
    logits,
    rtol=1e-4,
    atol=1e-5
)
print("Numerically close:", is_close)

**What is np.allclose()**?

It checks whether corresponding elements of two arrays are sufficiently close numerically.
We do not require perfect bit-for-bit equality, because different execution environments may use different optimizations for floating-point calculations.

Parameters:
- rtol - relative tolerance
- atol - absolute tolerance

> The tolerance values provided here are merely an exmaple of an experimental criterion and do not consitute a universal guarantee of the correctness of every export.

In [ ]:
pt_predictions = np.argmax(pt_logits, axis=-1)
onnx_predictions = np.argmax(logits, axis=-1)

print(
    "Predictions match:",
    np.array_equal(pt_predictions, onnx_predictions)
)

### Validation results

The exported ONNX model produces numerically consistent results with the original PyTorch model.
- Maximum absolute difference: 3.34e-06
- Mean absolute difference: 1.27e-06
- Numerical consistency check: Passed
- Prediction consistency check: Passed

Minor difference in logits are expected due to floating-point computations and differences between execution engines.

**Conclusion**: The ONNX model <b style="color:green">successfully</b> reproduces the PyTorch model's predictions for all tested inputs and <b style="color:green">is ready for the next stage</b>: deployment using NVIDIA Triton Inference Server.